# Multi-Scale HMM 01 — momentum / reversal states on 1h, 4h and 1D bars

Pre-registration: `discussions/Multi_Scale_HMM/001_claude_branch_preregistration.md`. Everything that decides the result is fixed in
`src/data_engineering/modeling/multiscale_hmm.py` (`registration()` below).

- **Bars:** UTC-aligned 1h / 4h / 1D bars built from Binance USD-M perpetual 1-minute klines; each bar carries intrabar emissions
  (standardized return, relative and absolute volatility, semivariance asymmetry, signed efficiency, close location, wick share,
  bipower jump share, signed taker flow, relative activity).
- **Model:** pooled multivariate Student-t HMM; $K$ and $\nu$ chosen on VALID; refitted on TRAIN+VALID and frozen.
- **Forecast:** per-state $y_{t+1}=a_k+b_k z_t$ ($b_k>0$ momentum, $b_k<0$ reversal), weighted by the filtered state probabilities.
- **Verdicts per scale:** S (statistical, sign-flip null with full refit, $\alpha=0.05/3$), E (net of 6 bp/side and funding, day-block
  bootstrap), states-add-value (vs $K=1$), and later P (prospective, data after 2026-10-18).

Environment: `MSH01_SCALES` (default `1h,4h,1D`), `MSH01_JOBS` (null worker processes, default CPU count − 1),
`MSH01_N_NULL` (override for a quick look only — the decision uses the registered counts), `MSH01_SYNTHETIC=1` (smoke run to a temp dir).

In [ ]:
from pathlib import Path
import json, os, sys, tempfile, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 250)
load_dotenv()
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = str(PROJECT_ROOT / 'src')
sys.path.insert(0, SRC)
os.environ['PYTHONPATH'] = SRC + (os.pathsep + os.environ['PYTHONPATH'] if os.environ.get('PYTHONPATH') else '')  # for null workers
from data_engineering.modeling import multiscale_hmm as mh

SYNTHETIC = os.getenv('MSH01_SYNTHETIC', '').strip() == '1'
SCALES = [s.strip() for s in os.getenv('MSH01_SCALES', '1h,4h,1D').split(',') if s.strip()]
JOBS = int(os.getenv('MSH01_JOBS', str(max(1, (os.cpu_count() or 2) - 1))))
N_NULL = {s: int(os.getenv('MSH01_N_NULL', str(mh.N_NULL[s]))) for s in SCALES}
SPEC = mh.Spec(restarts=2 if SYNTHETIC else mh.RESTARTS, null_restarts=2 if SYNTHETIC else mh.NULL_RESTARTS)
if SYNTHETIC:
    N_NULL = {s: min(N_NULL[s], 4) for s in SCALES}
    OUT = Path(tempfile.mkdtemp(prefix='msh01_'))
else:
    DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
    BIN_DIR = Path(os.getenv('BINANCE_UM_DIR') or DATA_ROOT.parent / 'Binance').expanduser().resolve()
    OUT = PROJECT_ROOT / 'notebooks' / 'Multi_Scale_HMM' / '01'
OUT.mkdir(parents=True, exist_ok=True)
REGISTERED_RUN = (not SYNTHETIC) and all(N_NULL[s] == mh.N_NULL[s] for s in SCALES)
print(mh.selftest())
print('scales', SCALES, '| null', N_NULL, '| jobs', JOBS, '| registered run' if REGISTERED_RUN else '| NOT the registered run', '| out', OUT)
print(json.dumps(mh.registration(SPEC), indent=1, default=mh.json_default))

## 1. Data: 1-minute perpetual klines and funding (truncated at the end of TEST; later data is the prospective window)

In [ ]:
MIN, FUND, cov = {}, {}, []
if SYNTHETIC:
    MIN = mh.synthetic_minutes(760, seed=1, planted=0.3)
    FUND = mh.synthetic_funding(MIN)
else:
    for a in SPEC.assets:
        p = BIN_DIR / f'{a}USDT_um_1m.parquet'
        if not p.exists():
            raise FileNotFoundError(f'{p} missing: run tools\\download_binance_um_klines.py')
        MIN[a] = mh.load_um_1m(p)
        fp = BIN_DIR / 'basis' / 'funding' / f'{a}USDT_funding.parquet'
        FUND[a] = mh.load_funding(fp) if fp.exists() else None
        if FUND[a] is None:
            print(f'WARNING {a}: no funding file at {fp} -> E cannot pass (tools\\download_binance_basis_data.py)')
for a in list(MIN):
    m = MIN[a]
    MIN[a] = m[m['open_time'] < mh.TEST_END - pd.Timedelta(minutes=1) + pd.Timedelta(seconds=1)].reset_index(drop=True)
    f = FUND.get(a)
    cov.append({'asset': a, 'first_minute': MIN[a]['open_time'].min(), 'last_minute': MIN[a]['open_time'].max(),
                'minutes': len(MIN[a]), 'funding_rows': 0 if f is None else int((f.index < mh.TEST_END).sum()),
                'funding_last': None if f is None else f.index.max()})
cov = pd.DataFrame(cov); cov.to_csv(OUT / '01_coverage.csv', index=False); display(cov)
assert (cov['first_minute'] <= mh.TRAIN_START + pd.Timedelta(days=15)).all(), 'data must start by mid-Oct 2024 (warm-up)'

## 2. Model selection on VALID ($K$, $\nu$), then frozen fit on TRAIN+VALID and the TEST forecasts

In [ ]:
PANELS, SEL, RES, STATES, ASSET_ROWS = {}, [], {}, [], []
for sc in SCALES:
    t0 = time.time()
    P = mh.build_panel(MIN, sc, FUND)
    PANELS[sc] = P
    sel = mh.select_model(P, sc, SPEC.restarts, SPEC.seed)
    SEL.append(sel)
    best = sel.sort_values('valid_ll_per_bar', ascending=False).iloc[0]
    K, nu = int(best['K']), float(best['nu'])
    r = mh.run_pipeline(P, sc, K, nu, SPEC.restarts, SPEC.seed)
    RES[sc] = r
    STATES.append(mh.state_table(r, P, sc))
    for a, v in r['per_asset'].items():
        ASSET_ROWS.append({'scale': sc, 'asset': a, **v})
    (OUT / f'01_frozen_model_{sc}.json').write_text(json.dumps({
        'scale': sc, 'model': r['model'].to_dict(), 'regression': r['reg'].to_dict('records'),
        'regression_k1': r['reg_k1'].to_dict('records'), 'standardization': r['stats'], 'version': mh.VERSION},
        indent=1, default=mh.json_default))
    print(f"{sc}: K={K} nu={nu}  S={r['S_stat_bp']:.3f} bp/bar ({r['S_assets_positive']}/4 assets >0)  "
          f"net {r['net_mean_bp_day']:.1f} bp/day [{r['net_lo_bp_day']:.1f}, {r['net_hi_bp_day']:.1f}]  {time.time()-t0:.0f}s")
SEL = pd.concat(SEL); SEL.to_csv(OUT / '01_selection.csv', index=False)
STATES = pd.concat(STATES); STATES.to_csv(OUT / '01_states.csv', index=False)
ASSET_ROWS = pd.DataFrame(ASSET_ROWS); ASSET_ROWS.to_csv(OUT / '01_per_asset.csv', index=False)
display(SEL.round(4)); display(STATES.round(3)); display(ASSET_ROWS.round(3))

## 3. Sign-flip null (full refit per draw, $K$ and $\nu$ fixed at the real-data choice)

In [ ]:
NULLS = {}
for sc in SCALES:
    t0 = time.time(); r = RES[sc]
    def prog(i, n, sc=sc, t0=t0):
        if i % 10 == 0 or i == n:
            print(f'  {sc} null {i}/{n}  {time.time()-t0:.0f}s', flush=True)
    NULLS[sc] = mh.run_null(MIN, sc, r['K'], r['nu'], N_NULL[sc], SPEC.null_restarts, SPEC.seed, JOBS, prog)
    NULLS[sc].assign(scale=sc).to_csv(OUT / f'01_null_{sc}.csv', index=False)

## 4. Verdicts

In [ ]:
rows = []
for sc in SCALES:
    r, v = RES[sc], mh.verdict(RES[sc], NULLS[sc])
    rows.append({'scale': sc, 'K': r['K'], 'nu': r['nu'], 'S_stat_bp_per_bar': r['S_stat_bp'],
                 'S_assets_positive': r['S_assets_positive'], 'k1_S_stat_bp': r['k1_S_stat_bp'],
                 'tsmom_S_stat_bp': r['tsmom_S_stat_bp'], 'drift_S_stat_bp': r['drift_S_stat_bp'],
                 'net_mean_bp_day': r['net_mean_bp_day'], 'net_lo_bp_day': r['net_lo_bp_day'],
                 'net_hi_bp_day': r['net_hi_bp_day'], 'cost_bp_day': r['net_cost_bp_day'],
                 'funding_bp_day': r['net_funding_bp_day'], 'E_assets_positive': r['E_assets_positive'],
                 'k1_net_mean_bp_day': r['k1_net_mean_bp_day'], 'incr_mean_bp_day': r['incr_mean_bp_day'],
                 'incr_lo_bp_day': r['incr_lo_bp_day'], 'funding_missing': r['funding_missing'], **v})
SUM = pd.DataFrame(rows); SUM.to_csv(OUT / '01_summary.csv', index=False); display(SUM.T)
decision = {'version': mh.VERSION, 'registered_run': REGISTERED_RUN, 'synthetic': SYNTHETIC,
            'per_scale': {r['scale']: {k: r[k] for k in ('S_verdict', 'S_p', 'E_verdict', 'states_add_value', 'scale_verdict')} for r in rows},
            'branch_verdict': ('CANDIDATES: ' + ','.join(r['scale'] for r in rows if r['scale_verdict'] == 'CANDIDATE_FOR_PROSPECTIVE'))
                              if any(r['scale_verdict'] == 'CANDIDATE_FOR_PROSPECTIVE' for r in rows) else 'NO_TRADEABLE_MOMENTUM_STATES'}
(OUT / '01_decision.json').write_text(json.dumps(decision, indent=1, default=mh.json_default))
(OUT / '01_manifest.json').write_text(json.dumps({'registration': mh.registration(SPEC), 'n_null_used': N_NULL,
                                                  'jobs': JOBS, 'coverage': cov.to_dict('records')}, indent=1, default=mh.json_default))
print(json.dumps(decision, indent=1, default=mh.json_default))

## 5. TEST equity curves (net rule, equal weight; HMM vs $K=1$) and null distributions

In [ ]:
fig, ax = plt.subplots(2, len(SCALES), figsize=(5.5 * len(SCALES), 7), squeeze=False)
for j, sc in enumerate(SCALES):
    r = RES[sc]
    pn = mh.portfolio(r['net_rule']); pb = mh.portfolio({a: mh.pnl_frame(r['detail'][a].assign(
        fund_next=PANELS[sc][a]['fund_next'], seg=PANELS[sc][a]['seg']), r['detail'][a]['mhat_k1'].to_numpy(),
        (mh.window_mask(PANELS[sc][a], mh.TEST_START, mh.TEST_END) & np.isfinite(PANELS[sc][a]['r_next'].to_numpy(float))))
        for a in r['detail']})
    ax[0, j].plot(pd.to_datetime(pn['bar_end']), 100 * pn['net'].cumsum(), label='HMM net')
    ax[0, j].plot(pd.to_datetime(pb['bar_end']), 100 * pb['net'].cumsum(), label='K=1 net', alpha=.7)
    ax[0, j].plot(pd.to_datetime(pn['bar_end']), 100 * pn['gross'].cumsum(), label='HMM gross', ls='--', alpha=.7)
    ax[0, j].set_title(f'{sc}: cumulative % of notional (TEST)'); ax[0, j].legend(); ax[0, j].axhline(0, c='k', lw=.5)
    ax[1, j].hist(NULLS[sc]['S_stat_bp'], bins=30, color='grey'); ax[1, j].axvline(r['S_stat_bp'], c='r')
    ax[1, j].set_title(f'{sc}: always-in gross bp/bar, null vs real')
plt.tight_layout(); plt.savefig(OUT / '01_equity_and_null.png', dpi=110); plt.show()